# 01 Segmentation and the filter manifest

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CoralsINPA/RedSeaCoralIdentifier/blob/main/notebooks/01_segment_and_filter.ipynb)

Run it in Colab on a **GPU runtime** (Runtime > Change runtime type > T4). Run the first cell, then the others in order.

**What it does.** (1) Says which photos still need a mask JSON (the CoralSCOP segmentation) and creates the missing ones only if you ask. (2) Checks the real CoralSCOP code on a few photos: it segments them into a scratch folder and compares the masks with the ones already saved for the same photos. (3) Loads the run's manifest (one row per mask, with the reason every rejected mask was dropped) and shows the filter funnel for landscape and portrait photos.

**Safe to run and re-run.** It reads the shared Drive and only creates new files. It never overwrites one. On the active run `iou60_sta55_pts32` (1,714 photos, every one with a mask JSON, and a saved manifest) it changes nothing in the run: the status reports 0 photos to segment, the manifest is loaded, and the check writes only to its own scratch run `results_all/segcheck_iou60_sta55_pts32/` and to a new folder under `<run>/reports/`.

| Step | What it does | Time | Reads | Writes |
|---|---|---|---|---|
| Setup | mounts Drive, updates the clone, installs `coralid[segment,viz]` | 3-5 min | GitHub | nothing on Drive |
| Config | loads the run config, checks the inputs | seconds | the run folder | nothing |
| 1. Status | counts the photos with and without a mask JSON; creates the missing JSONs only if `SEGMENT_MISSING = True` | seconds; hours if photos are missing (GPU, resumable) | the folder listings; the CoralSCOP checkpoint only when segmenting | nothing by default; `json/<stem>.json` for photos without one |
| 2. Check | segments `CHECK_N` photos (default 5, from `data_50`, with a portrait one if there is one) and compares them with the saved masks | about 2-4 min on a T4 (an estimate: the model loads once, then a few seconds per photo) | the photos, the checkpoint, the saved JSONs of those photos | the scratch run `results_all/segcheck_iou60_sta55_pts32/` and `<run>/reports/segcheck_<date>/` |
| 3. Manifest | loads `manifest/manifest.parquet`; builds it (about 2 h, resumable) only if the run has none | seconds when it exists | the manifest | nothing when it exists |
| 4. Funnel | the masks left after each filter, by photo orientation, and one photo with its masks colored by reject reason | seconds | the manifest, one photo and its JSON | nothing |

**The check's verdict.** SAM on another GPU or library version need not give the same pixels, so equality is not demanded. A saved mask counts as *found again* when a new mask overlaps it with IoU of 0.9 or more (one new mask per saved mask). The numbers are given for all masks and for the large ones (area of at least 25,000 px, the ones that survive the filters).

- **PASS:** at least 90% of the large saved masks and 80% of all of them are found again, and the new run has 0.8 to 1.25 times as many masks.
- **WARN:** a little outside that. Look at the per-photo table and the overlays before trusting new segmentation.
- **FAIL:** a photo has another frame size or could not be segmented, fewer than half of the large masks are found again, or the mask count is off by a factor of two. That means the code or the weights differ.

These limits are first guesses; revise them after the first real run.

**What to paste back to the next session:** the printed status, the check's verdict with its lines and the per-photo table, and the funnel table.

In [ ]:
import os
import subprocess
import sys

# ================================ SETTINGS: the only place to edit ================================
DATA_ROOT = "/content/drive/MyDrive/colab notebooks/CoralsMasking"  # the Drive folder CoralsMasking
BRANCH = "main"
REPO_URL = "https://github.com/CoralsINPA/RedSeaCoralIdentifier"
REPO_DIR = "/content/RedSeaCoralIdentifier"
EXTRAS = "segment,viz"
# ==================================================================================================

# The package cannot install itself before it exists. A clone left by an earlier session of this
# runtime is brought to the branch head BEFORE anything is imported from it.
if os.path.isdir(f"{REPO_DIR}/.git"):
    steps = [
        ["git", "-C", REPO_DIR, "fetch", "origin", BRANCH],
        ["git", "-C", REPO_DIR, "checkout", BRANCH],
        ["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH],
        ["git", "-C", REPO_DIR, "submodule", "update", "--init"],
    ]
else:
    steps = [["git", "clone", "--recurse-submodules", "--branch", BRANCH, REPO_URL, REPO_DIR]]
for step in steps:
    print("$", " ".join(step))
    subprocess.run(step, check=True)
sys.path.insert(0, f"{REPO_DIR}/src")  # enough to import coralid.colab (standard library only)

from coralid.colab import setup  # noqa: E402

env = setup(
    EXTRAS, data_root=DATA_ROOT, branch=BRANCH, repo_url=REPO_URL, repo_dir=REPO_DIR, update=False
)

In [ ]:
import logging

import torch

from coralid.config import load_config

# ================================ SETTINGS ================================
CHECK_N = 5  # photos for the segmentation check
CHECK_PHOTOS = "data_50"  # the photo folder to draw them from (under DATA_ROOT)
CHECK_PREFIX = "segcheck_"  # the scratch run's name prefix; use another one to redo the check
SEGMENT_MISSING = False  # True: create the mask JSONs of photos that have none (GPU, resumable)
SAMPLE_PHOTO = None  # a photo name for the overlay of step 4, or None for the one with most masks
# ==========================================================================

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
logging.getLogger("coralid").setLevel(logging.INFO)

cfg = load_config(f"{REPO_DIR}/configs/iou60_sta55_pts32.yaml")  # reads CORALID_DATA_ROOT
sam = cfg.sam
print(
    f"run {cfg.run_name} | SAM {sam.model_type}, points_per_side {sam.points_per_side}, "
    f"pred_iou {sam.pred_iou_thresh}, stability {sam.stability_score_thresh}, "
    f"min region {sam.min_mask_region_area} | long side {sam.max_side}"
)
for label, path in {
    "photos": cfg.images_dir,
    "check photos": cfg.data_root / CHECK_PHOTOS,
    "mask JSONs": cfg.json_dir,
    "checkpoint": cfg.checkpoint_path,
    "manifest": cfg.manifest_path,
}.items():
    print(f"  {label:13} {'ok' if path.exists() else 'not there yet':13} {path}")
print(
    "GPU:",
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (the check needs one)",
)

## 1. Which photos still need a mask JSON?

A photo whose JSON exists is never segmented again and its JSON is never rewritten, whatever it holds. To redo a photo, move its JSON aside first. An error while segmenting goes to `<run>/segmentation/segment.failures.jsonl` and writes no JSON, so the next run retries it. On the active run this cell only counts.

In [ ]:
from coralid.segmentation import list_images, plan_segmentation, run_segmentation

photos = list_images(cfg.images_dir)
plan = plan_segmentation(photos, cfg.json_dir)
print(
    f"{len(photos)} photos: {len(plan.done)} have a mask JSON (never rewritten), "
    f"{len(plan.todo)} need one"
)
if plan.collisions:
    print("two photos share a file stem; only the first is segmented:", plan.collisions)
if plan.todo and SEGMENT_MISSING:
    if not torch.cuda.is_available():
        raise RuntimeError("Segmenting needs a GPU runtime (Runtime > Change runtime type > T4).")
    result = run_segmentation(cfg, progress=True)
    print(
        f"wrote {len(result.written)} JSONs ({result.n_empty} with no masks); "
        f"{result.n_remaining} photos still without one; {len(result.failures)} failed"
    )
elif plan.todo:
    print("Set SEGMENT_MISSING = True to create them (a GPU job; it resumes where it stopped).")
else:
    print("Nothing to segment: every photo has its mask JSON.")

## 2. Does the real CoralSCOP reproduce the saved masks? (the segmentation check)

The real CoralSCOP code, run through `coralid`, has never run on the real photos. This segments a few of them into the scratch run `results_all/<CHECK_PREFIX><run name>/` (never into the active run's `json/`) and compares the new masks with the saved ones for the same photos: the frame size, the number of masks, and the masks themselves, matched by IoU. A photo already segmented in the scratch run is not segmented again, so a second run only repeats the comparison; use another `CHECK_PREFIX` to redo the check from scratch. The photos of `CHECK_PHOTOS` must be the same files as the ones the saved masks were made from; if the check finds none with a saved JSON, or says `frame differs`, point `CHECK_PHOTOS` at `data_all`.

In [ ]:
import time

from coralid.parity_segmentation import run_segmentation_check, write_segmentation_check

if not torch.cuda.is_available():
    raise RuntimeError(
        "The segmentation check needs a GPU runtime (Runtime > Change runtime type > T4); "
        "on a CPU it is far too slow."
    )
t0 = time.perf_counter()
check = run_segmentation_check(
    cfg, n=CHECK_N, photos_dir=CHECK_PHOTOS, prefix=CHECK_PREFIX, progress=True
)
print(f"\ndone in {time.perf_counter() - t0:.0f} s; scratch run: {check.config.run_dir}")
for line in check.comparison.report_lines(cfg.filters.min_area):
    print(line)
display(check.comparison.photos.round(3))
print("Saved (a new folder):", write_segmentation_check(check, cfg.reports_dir))

In [ ]:
from coralid.manifest import geometric_manifest
from coralid.masks import AnnotationStore
from coralid.viz import mask_overlay

photos = check.comparison.photos
compared = photos[photos["status"] == "compared"].groupby("orientation").head(1)
if compared.empty:
    print("No photo could be compared, so there is nothing to draw.")
for name in compared["image_name"]:  # one landscape and one portrait photo, if both were checked
    for title, run_cfg in (
        ("saved masks (the active run)", cfg),
        ("new masks (the scratch run)", check.config),
    ):
        masks = geometric_manifest(run_cfg.json_dir, cfg.filters, image_names=[name])
        store = AnnotationStore(run_cfg.json_dir)
        print(title)
        display(
            mask_overlay(masks, store, check.config.images_dir, name, color_col="stage1_reason")
        )

## 3. The manifest and the filter funnel

`build_manifest` loads `manifest/manifest.parquet` when it exists (after checking that it was made with the config's filter and crop settings) and never overwrites it. If the run has none, it builds one from the photos and the mask JSONs: about 2 hours, resumable, and written only when every photo is done.

In [ ]:
from coralid.jobs import build_manifest

built = build_manifest(cfg, progress=True)
manifest = built.manifest
print(
    f"{built.status}: {len(manifest)} masks on {manifest['image_name'].nunique()} photos, "
    f"{int(manifest['keep'].sum())} kept\n{built.path}"
)

In [ ]:
from coralid.funnel import filter_funnel
from coralid.reports import new_report_folder, save_figure
from coralid.viz import funnel_chart

funnel = filter_funnel(manifest)
print("Masks removed by each filter and left after it, by photo orientation:")
display(funnel)
chart = funnel_chart(funnel, title=f"Filter funnel of {cfg.run_name}")
display(chart)
figures = new_report_folder(cfg.reports_dir, "figures_01")  # a new folder for this run's pictures
print("Saved:", save_figure(chart, figures, "filter_funnel"))

In [ ]:
photo = SAMPLE_PHOTO or manifest["image_name"].value_counts().index[0]
print(f"{photo}: the masks, colored by the reason they were dropped ('kept' is a crop)")
overlay = mask_overlay(
    manifest, AnnotationStore(cfg.json_dir), cfg.images_dir, photo, color_col="reject_reason"
)
display(overlay)
print("Saved:", save_figure(overlay, figures, f"mask_reasons_{photo}"))

In [ ]:
from coralid.viz import photo_panels

# Same photo three ways, as in the last cell of weak_classification: the original, every mask
# SAM found (purple), and the masks' bounding boxes. Change SAMPLE_PHOTO in the settings cell.
panels = photo_panels(manifest, AnnotationStore(cfg.json_dir), cfg.images_dir, photo)
display(panels)
print("Saved:", save_figure(panels, figures, f"sam_masks_{photo}"))
kept = manifest[manifest["keep"]]
print(f"Only the {int((kept['image_name'] == photo).sum())} masks kept as crops:")
panels = photo_panels(kept, AnnotationStore(cfg.json_dir), cfg.images_dir, photo)
display(panels)
print("Saved:", save_figure(panels, figures, f"kept_masks_{photo}"))

## Next

- Paste the printed status, the check's lines and per-photo table, and the funnel table into the next session.
- Open `02_zero_shot.ipynb` for the classification baseline.